In [1]:
import numpy as np
import os
import rasterio
from pyproj import Transformer
from functools import lru_cache
import geopandas as gpd  

from config import RASTER_FEATURE_MAP, RASTER_CRS_EPSG


In [6]:
from raster_extract import extract_features
raw_features = extract_features(19.8136,85.8315)
print(raw_features)

[Fallback] rainfall: using 1751.7821050088762
[Fallback] ext_rainfall: using 31.655798
{'distance_to_river': 1839.809814453125, 'aspect': 117.15594482421875, 'dem': 8.062529563903809, 'flow_accumulation': 2.0, 'twi': 10.992614207226595, 'slope': 0.3180866837501526, 'rainfall': 1751.7821050088762, 'drainage_density': 0.03337519243359566, 'ext_rainfall': 31.655798, 'lulc': 50.0, 'soil': 31.0, 'easting': 377618.9795778657, 'northing': 2191277.539657772}


In [7]:
from preprocess import preprocess_features
from predict import predict

In [8]:
scaled = preprocess_features(raw_features)
pred = predict(scaled) 

c:\Users\ps302\OneDrive\Desktop\Hydrology\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [9]:
scaled

,distance_to_river,aspect,dem,flow_accumulation,twi,slope,rainfall,drainage_density,ext_rainfall,lulc,soil
0,1839.809814,117.155945,8.06253,2.0,10.992614,0.318087,1751.782105,0.033375,31.655798,50.0,31.0


In [12]:

pred["prediction"], pred["probability"]

(1, 0.7565239642525363)

In [ ]:

prob = float(model.predict_proba(model_input)[:, 1][0])
pred = int(prob >= threshold)

In [15]:
# Coordinate transformers (cached) 
@lru_cache(maxsize=1)
def _get_transformer_to_utm():
    """EPSG:4326 (lat/lon) → EPSG:32645 (UTM 45N) for raster sampling."""
    return Transformer.from_crs("EPSG:4326", f"EPSG:{RASTER_CRS_EPSG}", always_xy=True)

In [5]:
def _latlon_to_utm(lat: float, lon: float) -> tuple[float, float]:
    """Convert lat/lon to UTM zone 45N coordinates."""
    transformer = _get_transformer_to_utm()
    x, y = transformer.transform(lon, lat)  # always_xy: input is (lon, lat)
    return x, y

In [ ]:
def _sample_raster(raster_path: str, x: float, y: float) -> float | None:
    """
    Sample a single pixel value from a raster at the given projected coordinates.

    Returns None if the point is outside the raster extent or the value is nodata.
    """
    try:
        with rasterio.open(raster_path) as src:
            # Convert projected coordinate to pixel row/col
            row, col = src.index(x, y)

            # Bounds check
            if row < 0 or row >= src.height or col < 0 or col >= src.width:
                return None

            # Read the pixel value (band 1)
            val = src.read(1, window=rasterio.windows.Window(col, row, 1, 1))
            val = float(val[0, 0])

            # Check nodata
            if src.nodata is not None and np.isclose(val, src.nodata):
                return None

            # Treat NaN / Inf as missing data
            if not np.isfinite(val):
                return None

            return val
    except Exception:
        return None

In [21]:
def sample_all_rasters(x: float, y: float, rasters: dict[str, str]) -> dict | None:
    """
    Sample all configured rasters at projected coordinates (x, y).

    Returns a dictionary of feature values, or None if any raster is missing,
    out of bounds, nodata, or invalid for this location.
    """
    values = {}

    for feature_name, raster_path in rasters.items():
        if not os.path.exists(raster_path):
            print(f"Warning: Missing {feature_name} at {raster_path}")
            return None

        try:
            with rasterio.open(raster_path) as src:
                in_bounds = (
                    src.bounds.left <= x <= src.bounds.right
                    and src.bounds.bottom <= y <= src.bounds.top
                )
                if not in_bounds:
                    return None

                val = list(src.sample([(x, y)]))[0][0]

                if not np.isfinite(val):
                    return None

                if src.nodata is not None and np.isclose(val, src.nodata):
                    return None

                values[feature_name] = float(val)
        except Exception:
            return None

    return values

In [7]:
# RASTER_FEATURE_MAP

In [33]:
def extract_features(lat: float, lon: float) -> dict | None:
    """
    Extract all raster feature values for a lat/lon point.

    Parameters
    ----------
    lat : float
        Latitude (WGS84).
    lon : float
        Longitude (WGS84).

    Returns
    -------
    dict or None
        Dictionary with feature names and sampled values.
        Includes 'easting' and 'northing' in target raster CRS.
        Returns None if any raster has missing/invalid data at that point.
    """
    utm_x, utm_y = _latlon_to_utm(lat, lon)
    print(utm_x, utm_y)

    features = {}
    for feature_name, raster_path in RASTER_FEATURE_MAP.items():
        print("=" * 55, "/n")
        print(feature_name, raster_path)
        if not os.path.exists(raster_path):
            print(f"Warning: Missing {feature_name} at {raster_path}")
            return None
      
        val = _sample_raster(raster_path, utm_x, utm_y)
        print(f"VALUE: {val}")
        # if val is None:
        #     return None
        features[feature_name] = float(val)

    features["easting"] = float(utm_x)
    features["northing"] = float(utm_y)
    return features

In [37]:
data = extract_features(19.8136,85.8315)



[Fallback] rainfall: using 1751.7821050088762
[Fallback] ext_rainfall: using 31.655798


In [35]:
import pandas as pd
import numpy as np
import os

# Build fallback values from training data (more realistic than hard-coded zeros)
TRAINING_CSV = r"C:\Users\ps302\OneDrive\Desktop\Hydrology\src\Flood_Model\data\proceessed\flood_training_data_10k_clean.csv"

FEATURE_ORDER = [
    "distance_to_river",
    "aspect",
    "dem",
    "flow_accumulation",
    "twi",
    "slope",
    "rainfall",
    "drainage_density",
    "ext_rainfall",
    "lulc",
    "soil",
]

def build_safe_defaults(training_csv: str) -> dict[str, float]:
    if not os.path.exists(training_csv):
        # last-resort defaults if csv is unavailable
        return {f: 0.0 for f in FEATURE_ORDER}

    df_ref = pd.read_csv(training_csv)
    defaults = {}

    for f in FEATURE_ORDER:
        if f not in df_ref.columns:
            defaults[f] = 0.0
            continue

        s = df_ref[f].dropna()
        if s.empty:
            defaults[f] = 0.0
            continue

        # Categorical-like fields: use mode
        if f in ("lulc", "soil"):
            defaults[f] = float(s.mode().iloc[0])
        else:
            # Numeric fields: use median
            defaults[f] = float(s.median())

    return defaults

SAFE_DEFAULTS = build_safe_defaults(TRAINING_CSV)


def extract_features(lat: float, lon: float) -> dict | None:
    utm_x, utm_y = _latlon_to_utm(lat, lon)

    features = {}
    for feature_name, raster_path in RASTER_FEATURE_MAP.items():
        if not os.path.exists(raster_path):
            # missing raster -> fallback
            features[feature_name] = SAFE_DEFAULTS.get(feature_name, 0.0)
            continue

        val = _sample_raster(raster_path, utm_x, utm_y)

        # instead of returning None, inject safe fallback to avoid pipeline failure
        if val is None or not np.isfinite(val):
            fallback = SAFE_DEFAULTS.get(feature_name, 0.0)
            print(f"[Fallback] {feature_name}: using {fallback}")
            features[feature_name] = fallback
        else:
            features[feature_name] = float(val)

    features["easting"] = float(utm_x)
    features["northing"] = float(utm_y)
    return features

In [36]:
data = extract_features(19.8136,85.8315)

[Fallback] rainfall: using 1751.7821050088762
[Fallback] ext_rainfall: using 31.655798


In [16]:
BASIN_SHP_PATH = r"C:\Users\ps302\OneDrive\Desktop\Hydrology\data\basin_boudary\mahanadi_basin.shp"
BASE_DIR = r"C:\Users\ps302\OneDrive\Desktop\Hydrology"
DATA_DIR = os.path.join(BASE_DIR, "data")
RASTER_DIR = os.path.join(DATA_DIR, "Flood_data_final")
print(RASTER_DIR)

C:\Users\ps302\OneDrive\Desktop\Hydrology\data\Flood_data_final


In [17]:
# STEP 1: Raster Feature Map
RASTER_FEATURE_MAP = {
    "distance_to_river": os.path.join(RASTER_DIR, "distance_to_river.tif"),
    "aspect": os.path.join(RASTER_DIR, "aspect.tif"),
    "dem": os.path.join(RASTER_DIR, "dem_30.tif"),
    "flow_accumulation": os.path.join(RASTER_DIR, "flow_acc_30m.tif"),
    "twi": os.path.join(RASTER_DIR, "TWI.tif"),
    "slope": os.path.join(RASTER_DIR, "fixed_slope.tif"), 
    "rainfall": os.path.join(RASTER_DIR, "rainfall_30m_f.tif"),
    "drainage_density": os.path.join(RASTER_DIR, "drainage_density_30.tif"), 
    "ext_rainfall": os.path.join(RASTER_DIR, "ext_rainfall.tif"),
    "lulc": os.path.join(RASTER_DIR, "lulc_30m.tif"),
    "soil": os.path.join(RASTER_DIR, "soil_clay_30m.tif"),
}

In [18]:
# LOAD & ALIGN SHAPEFILE (Fix CRS)
print("Loading Basin Shapefile...")
basin_gdf = gpd.read_file(BASIN_SHP_PATH)

print(" Checking Raster CRS...")
# Open DEM just to get the correct UTM projection
with rasterio.open(RASTER_FEATURE_MAP["dem"]) as src:
    raster_crs = src.crs

# Reproject Shapefile to match Raster CRS (Crucial Step!)
if basin_gdf.crs != raster_crs:
    print(f" Reprojecting basin from {basin_gdf.crs} to {raster_crs}...")
    basin_gdf = basin_gdf.to_crs(raster_crs)

# Combine multi-polygons into a single geometry for checking
basin_geom = basin_gdf.geometry.unary_union
minx, miny, maxx, maxy = basin_geom.bounds

Loading Basin Shapefile...
 Checking Raster CRS...
 Reprojecting basin from PROJCS["WGS_1984_Lambert_Conformal_Conic",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",24],PARAMETER["central_meridian",80],PARAMETER["standard_parallel_1",12.4729444],PARAMETER["standard_parallel_2",35.17280555],PARAMETER["false_easting",4000000],PARAMETER["false_northing",4000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]] to EPSG:32645...


C:\Users\ps302\AppData\Local\Temp\ipykernel_24632\667160961.py:16: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  basin_geom = basin_gdf.geometry.unary_union


In [ ]:
import os

#  Base paths
BASE_DIR = r"C:\Users\ps302\OneDrive\Desktop\Hydrology"

In [ ]:
def extract_features(lat: float, lon: float) -> dict | None:
    """
    Extract all raster feature values for a lat/lon point.

    Parameters
    ----------
    lat : float
        Latitude (WGS84).
    lon : float
        Longitude (WGS84).

    Returns
    -------
    dict or None
        Dictionary with feature names and sampled values.
        Includes 'easting' and 'northing' in target raster CRS.
        Returns None if any raster has missing/invalid data at that point.
    """
    utm_x, utm_y = _latlon_to_utm(lat, lon)

    features = {}
    for feature_name, raster_path in RASTER_FEATURE_MAP.items():
        if not os.path.exists(raster_path):
            print(f"Warning: Missing {feature_name} at {raster_path}")
            return None

        val = _sample_raster(raster_path, utm_x, utm_y)
        if val is None:
            return None
        features[feature_name] = float(val)

    features["easting"] = float(utm_x)
    features["northing"] = float(utm_y)
    return features

In [61]:
data = extract_features(19.8136,85.8315)



377618.9795778657 2191277.539657772
hello world
rainfall C:\Users\ps302\OneDrive\Desktop\Hydrology\data\Flood_data_final\rainfall_30m_f.tif 

VALUE: None


In [62]:
os.path.join(RASTER_DIR, "rainfall_30m_f.tif")

'C:\\Users\\ps302\\OneDrive\\Desktop\\Hydrology\\data\\Flood_data_final\\rainfall_30m_f.tif'

In [87]:
import rasterio
from pyproj import Transformer
raster_path = os.path.join(RASTER_DIR, "dem_30.tif")

with rasterio.open(raster_path) as src:
    raster_crs = src.crs

transformer = Transformer.from_crs("EPSG:4326", raster_crs, always_xy=True)

# Transformer (lat/lon → UTM)
# transformer = Transformer.from_crs("EPSG:4326", "EPSG:32645", always_xy=True)

def get_raster_value(raster_path, lat, lon):
    # Convert lat/lon → UTM
    # x, y = transformer.transform(lon, lat)
    x, y = 4440315.488, 3795031.364
    print(x, y)

    with rasterio.open(raster_path) as src:
        # Sample raster at (x, y)
        value = list(src.sample([(x, y)]))[0][0]

    return value


# Example


print(raster_path)
lat, lon = 19.8136, 85.8315

val = get_raster_value(raster_path, lat, lon)
print("Raster value:", val)

C:\Users\ps302\OneDrive\Desktop\Hydrology\data\Flood_data_final\dem_30.tif
4440315.488 3795031.364
Raster value: nan


In [86]:
import rasterio

raster_path = os.path.join(RASTER_DIR, "dem_30.tif")
with rasterio.open(raster_path) as src:
    print(src.crs)
    print(src.bounds)

EPSG:32645
BoundingBox(left=-191076.69514340186, bottom=2130778.5796351717, right=480803.30485659814, top=2621158.5796351717)


In [74]:
import rasterio
from pyproj import Transformer
import numpy as np

transformer = Transformer.from_crs("EPSG:4326", "EPSG:32645", always_xy=True)

def get_raster_value(raster_path, lat, lon):
    x, y = transformer.transform(lon, lat)   # correct conversion
    with rasterio.open(raster_path) as src:
        in_bounds = (src.bounds.left <= x <= src.bounds.right) and (src.bounds.bottom <= y <= src.bounds.top)
        if not in_bounds:
            return None, "out_of_bounds"

        val = list(src.sample([(x, y)]))[0][0]
        if not np.isfinite(val):
            return None, "nan_pixel"

        if src.nodata is not None and np.isclose(val, src.nodata):
            return None, "nodata"

    return float(val), "ok"

raster_path = os.path.join(RASTER_DIR, "rainfall_30m_f.tif")
print(raster_path)
lat, lon = 19.8136, 85.8315

val = get_raster_value(raster_path, lat, lon)
print(val)

C:\Users\ps302\OneDrive\Desktop\Hydrology\data\Flood_data_final\rainfall_30m_f.tif
(None, 'nan_pixel')


In [26]:
from pyproj import Transformer

# Lat/Lon
lat, lon = 19.8136, 85.8315

# WGS84 → UTM Zone 45N
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32645", always_xy=True)

easting, northing = transformer.transform(lon, lat)

print("Easting:", easting)
print("Northing:", northing)

Easting: 377618.9795778657
Northing: 2191277.539657772


In [28]:
from pyproj import Transformer
from functools import lru_cache

@lru_cache(maxsize=1)
def get_transformer():
    print("Creating transformer...")  # to show effect
    return Transformer.from_crs("EPSG:4326", "EPSG:32645", always_xy=True)

def convert(lat, lon):
    transformer = get_transformer()
    x, y = transformer.transform(lon, lat)
    return x, y

# Multiple calls
print(convert(22.3, 87.2))
print(convert(22.4, 87.3))
print(convert(22.5, 87.4))

Creating transformer...
(520600.4962546758, 2466047.004284649)
(530878.7610747673, 2477133.2402978856)
(541142.2744152668, 2488226.6034048484)


In [27]:
from pyproj import Transformer

def convert(lat, lon):
    transformer = Transformer.from_crs("EPSG:4326", "EPSG:32645", always_xy=True)
    x, y = transformer.transform(lon, lat)
    return x, y

# Call multiple times
print(convert(22.3, 87.2))
print(convert(22.4, 87.3))
print(convert(22.5, 87.4))

(520600.4962546758, 2466047.004284649)
(530878.7610747673, 2477133.2402978856)
(541142.2744152668, 2488226.6034048484)
